In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze

In [0]:
# Day 1 — Bronze Layer: synthetic data generation + raw ingestion to Delta
#
# Run this in a Databricks Community Edition notebook (recommended),
# or locally with: pip install pyspark delta-spark --break-system-packages
#
# Concept practiced: cluster/driver/executor model, lazy evaluation,
# partitioning — you'll see all three in action below.

import random
from datetime import date, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id, countDistinct
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, DateType
)

# --- 1. SparkSession (the driver's entry point) ---
spark = SparkSession.builder.appName("merchant-campaign-bronze").getOrCreate()

# --- 1b. Ensure the target schema exists (Unity Catalog: catalog.schema.table) ---
# Databricks Free Edition's default catalog is called "workspace".
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# --- 2. Define schema explicitly (no lazy schema inference in production pipelines) ---
schema = StructType([
    StructField("merchant_id", StringType(), False),
    StructField("campaign_id", StringType(), False),
    StructField("market", StringType(), False),
    StructField("date", DateType(), False),
    StructField("spend", DoubleType(), True),
    StructField("impressions", IntegerType(), True),
    StructField("clicks", IntegerType(), True),
])

# --- 3. Generate synthetic campaign event data ---
MARKETS = ["DE", "NL", "PL", "IT", "FR"]
MERCHANTS = [f"m_{i:04d}" for i in range(1, 51)]     # 50 merchants
CAMPAIGNS = [f"c_{i:04d}" for i in range(1, 201)]    # 200 campaigns

def random_date(start: date, end: date) -> date:
    delta_days = (end - start).days
    return start + timedelta(days=random.randint(0, delta_days))

def generate_rows(n: int):
    start, end = date(2026, 1, 1), date(2026, 6, 30)
    for _ in range(n):
        impressions = random.randint(100, 50000)
        # click-through rate roughly 0.5% - 4%, with some noise
        clicks = int(impressions * random.uniform(0.005, 0.04))
        spend = round(clicks * random.uniform(0.15, 1.2), 2)  # cost per click model
        yield (
            random.choice(MERCHANTS),
            random.choice(CAMPAIGNS),
            random.choice(MARKETS),
            random_date(start, end),
            spend,
            impressions,
            clicks,
        )

# Nothing has actually executed yet up to this point beyond Python-side generation.
rows = list(generate_rows(20000))

# --- 4. Build a DataFrame (still lazy — no Spark job has run) ---
df = spark.createDataFrame(rows, schema=schema)

# Check partition count before the write triggers execution.
# NOTE: df.rdd.getNumPartitions() is NOT available on serverless compute —
# serverless uses Spark Connect, which doesn't expose direct RDD/JVM access.
# spark_partition_id() works through the DataFrame API instead, so it's
# compatible with both serverless and classic compute.
initial_partitions = (
    df.select(spark_partition_id().alias("pid"))
    .agg(countDistinct("pid"))
    .collect()[0][0]
)
print(f"Initial partitions: {initial_partitions}")

# --- 5. Repartition by market (matches how we'll query/serve later) ---
# This is a deliberate choice tied to the concept block: partitioning by a
# low-cardinality, frequently-filtered column enables partition pruning
# on read, and keeps write file sizes reasonable per partition.
df = df.repartition("market")

# --- 6. Write Bronze layer as a Delta table, partitioned by market ---
# THIS is the action that finally triggers execution across the cluster.
(
    df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("market")
    .saveAsTable("workspace.bronze.merchant_campaign_events")
)

print("Bronze ingestion complete.")
df.groupBy("market").count().show()